In [ ]:
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from scipy.stats import spearmanr
from scipy.special import softmax,xlogy

from sklearn.metrics import make_scorer
from sklearn.metrics.pairwise import cosine_similarity as cosim
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline

# Importing the dataset

In [ ]:
# import the aligned dataset
aligned  = pd.read_pickle('data/en80jours_aligned.pkl')
print(aligned.columns)
aligned.sample(3)

In [ ]:
# terms with freq > 5
frequent_fr_norm = [term for term, freq in aligned.fr_norm.value_counts().items() if freq > 5]
frequent_en_norm = [term for term, freq in aligned.en_norm.value_counts().items() if freq > 5]
frequent_de_norm = [term for term, freq in aligned.de_norm.value_counts().items() if freq > 5]
frequent_sb_norm = [term for term, freq in aligned.sr_norm.value_counts().items() if freq > 5]

# keep only rows where all 4 terms are in the frequent lists
print(f"Original size: {len(aligned)}")
aligned_frequent = aligned[
    aligned.fr_norm.isin(frequent_fr_norm) &
    aligned.en_norm.isin(frequent_en_norm) &
    aligned.de_norm.isin(frequent_de_norm) &
    aligned.sr_norm.isin(frequent_sb_norm)
]
print(f"Size after filtering: {len(aligned_frequent)}")

# keep only rows where all 4 terms are not None or empty
aligned_frequent = aligned_frequent[
    aligned_frequent.fr_norm.notna() & (aligned_frequent.fr_norm != '') & (aligned_frequent.fr_norm != 'None') &
    aligned_frequent.en_norm.notna() & (aligned_frequent.en_norm != '') & (aligned_frequent.en_norm != 'None') &
    aligned_frequent.de_norm.notna() & (aligned_frequent.de_norm != '') & (aligned_frequent.de_norm != 'None') &
    aligned_frequent.sr_norm.notna() & (aligned_frequent.sr_norm != '') & (aligned_frequent.sr_norm != 'None')
]
print(f"Size after removing empty/None: {len(aligned_frequent)}")

# 1) Modelling the psychological space of spatial relations

## Importing pile sorting results

$N=35$ participants, $K=30$ cards to sort

In [ ]:
N = 35
K = 30

In [ ]:
all_exp = [ 
    pd.read_csv(f'pile_sorting_results/subject_{i}_piles.csv')
    for i in range(N)
]
nb_piles  = [piles.shape[1] for piles in all_exp]

# distribution of number of piles
plt.figure(figsize=(4, 2))
plt.hist(nb_piles, bins=range(1, max(nb_piles)+1), align='left', rwidth=0.8)
plt.xlabel('Number of Piles')
plt.ylabel('Count')
plt.show()

In [ ]:
from regex import M


def piles_to_matrix(exp: pd.DataFrame) -> np.ndarray:
    """Convert pile-sorting dataframe to a count matrix"""
    n_items = K
    count_matrix = np.zeros((n_items, n_items))

    for col in exp.columns:
        for i in exp[col].dropna():
            for j in exp[col].dropna():
                count_matrix[int(i), int(j)] = 1
                count_matrix[int(j), int(i)] = 1

    assert np.allclose(count_matrix, count_matrix.T), "Count matrix is not symmetric"

    return count_matrix

count_matrices = [piles_to_matrix(exp) for exp in all_exp]
aggregated_matrix = np.mean(count_matrices, axis=0)
np.fill_diagonal(aggregated_matrix, 1)

# visualize the aggregated similarity matrix as a heatmap
plt.figure()
sns.heatmap(aggregated_matrix, cmap='viridis', cbar=True, linewidth=.5)
plt.show()


## Training a model to predict the similarities $\widehat{sim}_{i,j}$

- human similarities $sim_{i,j}$ : `human_sim`
- contextual embeddings $F_{i,j}$ : `F_pairs`

In [ ]:
# human pairwise similarities
human_sim = []
for i in range(30):
    for j in range(i+1, 30):
        human_sim.append(aggregated_matrix[i, j])
human_sim = np.array(human_sim)

In [ ]:
centroids_embeddings_list = pickle.load(open('data/centroids_embeddings_list.pkl', 'rb'))

# pairs of input embeddings
F_pairs = []
for i in range(len(centroids_embeddings_list)):
    for j in range(i+1, len(centroids_embeddings_list)):
        F_pairs.append((centroids_embeddings_list[i], centroids_embeddings_list[j]))
F_pairs = np.array(F_pairs)

### baseline : cosine similarity

In [ ]:
sim_hat = [
    cosim(F_pairs[i][0].reshape(1, -1), F_pairs[i][1].reshape(1, -1))[0][0]
    for i in range(len(F_pairs))
]


corr, p_value = spearmanr(human_sim, sim_hat)
print(f"Spearman correlation: {corr:.4f}, p-value: {p_value:.4e}")

### Ridge regression

In [ ]:
# Custom Transformer for Interaction Features
class EmbeddingInteractions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X shape: (n_samples, 2, embedding_dim)
        u, v = X[:, 0, :], X[:, 1, :]
        
        # Features: Hadamard (alignment) and L1 (distance)
        prod = u * v
        diff = np.abs(u - v)
        
        # Cosine similarity
        norm_u = np.linalg.norm(u, axis=1, keepdims=True)
        norm_v = np.linalg.norm(v, axis=1, keepdims=True)
        # Add small epsilon to avoid division by zero
        cosine = np.sum(u * v, axis=1, keepdims=True) / (norm_u * norm_v + 1e-10)
        
        return np.hstack([prod, diff, cosine])

def spearman_metric(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

# Pipeline Definition
pipe = Pipeline([
    ('features', EmbeddingInteractions()),
    ('scaler', StandardScaler()),
    ('model', Ridge(solver='svd')) 
])

# Nested Cross-Validation Setup
# Inner CV: For hyperparameter optimization
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Outer CV: For performance evaluation
outer_cv = KFold(n_splits=5, shuffle=True, random_state=43)

# Parameter grid for Ridge alpha (regularization strength)
param_grid = {
    'model__alpha': np.logspace(-3, 3, 10) # Test values from 0.001 to 1000
}

# GridSearch 
clf = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=inner_cv,
    scoring=make_scorer(spearman_metric),
    n_jobs=-1
)

# Evaluation
nested_scores = cross_val_score(
    clf, 
    F_pairs, 
    human_sim, 
    cv=outer_cv, 
    scoring=make_scorer(spearman_metric),
    #n_jobs=-1
)

print(f"Nested CV Mean Spearman Rho: {np.mean(nested_scores):.3f} (±{np.std(nested_scores):.3f})")


### low-rank projection model

In [ ]:
class PairedStandardScaler(BaseEstimator, TransformerMixin):
    """Custom scaler MinMaxScaler"""
    def __init__(self):
        self.scaler = MinMaxScaler()
        
    def fit(self, X: np.ndarray, y=None):
        # Flatten (N_pairs, 2, N_features) -> (N_pairs * 2, N_features)
        self.scaler.fit(X.reshape(-1, X.shape[-1]))
        return self
        
    def transform(self, X: np.ndarray) -> np.ndarray:
        scaled_flat = self.scaler.transform(X.reshape(-1, X.shape[-1]))
        return scaled_flat.reshape(X.shape)

class low_rank_proj(BaseEstimator, RegressorMixin):
    def __init__(self, bottleneck_dim: int = 5, lambda_: float = 0.1, 
                 max_iter: int = 1000, lr: float = 0.001, clip_norm: float = 5.0):
        self.bottleneck_dim = bottleneck_dim
        self.lambda_ = lambda_
        self.max_iter = max_iter
        self.lr = lr
        self.clip_norm = clip_norm
        self.W_B = None
    
    def fit(self, F_pairs: np.ndarray, s: np.ndarray):
        n_pairs, n_features = F_pairs.shape[0], F_pairs.shape[2]
        rng = np.random.RandomState(42)
        
        # Initialize projection matrix
        self.W_B = rng.randn(self.bottleneck_dim, n_features) * 0.01
        
        prev_loss = np.inf
        
        for epoch in range(self.max_iter):
            L_total = 0.0
            grad = np.zeros_like(self.W_B)
            
            # Vectorized forward pass
            for idx in range(n_pairs):
                F_i = F_pairs[idx, 0]
                F_j = F_pairs[idx, 1]
                s_ij = s[idx]
                
                # Projections
                B_Fi = self.W_B @ F_i
                B_Fj = self.W_B @ F_j
                s_hat_ij = B_Fi.T @ B_Fj
                
                # Loss accumulation
                error = s_hat_ij - s_ij
                L_total += error ** 2
                
                # Gradient accumulation
                grad += 2 * error * (np.outer(B_Fj, F_i) + np.outer(B_Fi, F_j))
            
            # Regularization and Loss Averaging
            L_total = L_total / 2 + self.lambda_ * np.sum(self.W_B ** 2)
            grad = grad / n_pairs + 2 * self.lambda_ * self.W_B
            
            # Convergence check
            if abs(prev_loss - L_total) < 1e-6:
                break
            prev_loss = L_total
            
            # Gradient Clipping
            grad_norm = np.linalg.norm(grad)
            if np.isfinite(grad_norm) and grad_norm > self.clip_norm:
                grad *= self.clip_norm / (grad_norm + 1e-12)
            
            self.W_B -= self.lr * grad
            
        return self
    
    def predict(self, F_pairs: np.ndarray) -> np.ndarray:
        if self.W_B is None:
            raise RuntimeError("Model must be fit before prediction.")
            
        s_hat = []
        for idx in range(F_pairs.shape[0]):
            F_i = F_pairs[idx, 0]
            F_j = F_pairs[idx, 1]
            B_Fi = self.W_B @ F_i
            B_Fj = self.W_B @ F_j
            s_hat.append(B_Fi.T @ B_Fj)
        return np.array(s_hat)
    
def spearman_metric(y_true, y_pred):
    return spearmanr(y_true, y_pred).correlation

spearman_score = make_scorer(spearman_metric, greater_is_better=True)

In [ ]:
# CV pipeline
pipe = Pipeline([
    ('scaler', PairedStandardScaler()),
    ('simdr', low_rank_proj())
])

# 1. Define the dimensions to analyze
dims_to_test = [5] #[1,2,3,4,5,6,7,8,9,10,20,50,100]

print(f"{'Dim':<6} | {'Mean Spearman':<15} | {'Std Dev':<10}")
print("-" * 38)

for d in dims_to_test:
    
    # grid for INNER optimization
    param_grid_inner = {
        'simdr__lambda_': [0.1], # [0.001, 0.01, 0.1, 1.0, 10, 100],
        'simdr__lr': [0.001], # [0.001, 0.01, 0.1],
        'simdr__bottleneck_dim': [d]  # Fixed for this run
    }

    # Inner CV: Finds best lambda/lr for this specific d
    # This prevents training data leakage into hyperparameter selection
    inner_cv = KFold(n_splits=6, shuffle=True, random_state=42)
    
    clf = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid_inner,
        cv=inner_cv,
        scoring=spearman_score,
        n_jobs=-1,
        verbose=0
    )
    
    # Outer CV: Estimates generalization error
    # Evaluates the "process of tuning a d-dimensional model" on unseen data
    outer_cv = KFold(n_splits=6, shuffle=True, random_state=43)
    
    nested_scores = cross_val_score(
        clf, 
        F_pairs, 
        human_sim, 
        cv=outer_cv, 
        scoring=spearman_score,
        n_jobs=-1
    )
    
    # Record metrics
    mean_score = np.mean(nested_scores)
    std_score = np.std(nested_scores)
    
    print(f"{d:<6} | {mean_score:.4f}          | {std_score:.4f}")

In [ ]:
# Pipeline with Best Hyperparameters
final_pipe = Pipeline([
    ('scaler', PairedStandardScaler()),
    ('simdr', low_rank_proj(
        bottleneck_dim=5, 
        lambda_=0.1, 
        lr=0.001
    ))
])

# Train on all available data
print("Training final model on full dataset...")
final_pipe.fit(F_pairs, human_sim)

## predictions of similarities on the whole dataset

In [ ]:
# predicted_sim is the matrix of predicted similarities for all pairs of terms, 
# computed using the final model.
size = len(aligned_frequent)
predicted_sim = np.zeros((size, size))
for i in range(size):
    for j in range(i+1, size):
        f_i = aligned_frequent.iloc[i]['embeddings']
        f_j = aligned_frequent.iloc[j]['embeddings']
        pair = np.array([[f_i, f_j]])
        predicted_sim[i, j] = final_pipe.predict(pair)[0]
        predicted_sim[j, i] = predicted_sim[i, j]  # Symmetric matrix

In [ ]:
# normalize each row with a softmax
predicted_sim_normalized = np.zeros_like(predicted_sim)
for i in range(size):
    predicted_sim_normalized[i] = softmax(predicted_sim[i])

# does each row sum to 1? boolean check
row_sums = predicted_sim_normalized.sum(axis=1)
print("All rows sum to 1:", np.allclose(row_sums, 1.0))

# display heatmap of predicted similarities
plt.figure()
sns.heatmap(predicted_sim_normalized, cmap='viridis', cbar=True)
plt.show()

# 2) Building alignment tables

In [ ]:
# to alignement tables 
pairs = [('fr', 'en'), ('fr', 'de'), ('fr', 'sr')]
tables = []
dataframes = []
data = aligned_frequent.reset_index(drop=True)

for source, target in pairs:
    lexicon = data[f'{target}_norm'].unique()
    source_terms = data[f'{source}_norm']
    ids = data['hash']
    source_hash = [f"{term}_{ID}" for ID,term in zip(ids, source_terms)]
    
    translations = np.zeros((len(data), len(lexicon)), dtype=int)
    print(translations.shape)

    for i,row in data.iterrows():
        target_term = row[f'{target}_norm']
        if target_term in lexicon:
            j = lexicon.tolist().index(target_term)
            translations[i,j] += 1
        else :
            print(f"Skipping target term: {target_term}")
            pass

    tables.append(translations)
    dataframes.append(pd.DataFrame(translations, index=source_hash, columns=lexicon))

# assert each row sums to 1
for df in dataframes:
    assert all(df.sum(axis=1) == 1)

dataframes[0]

# 3) LLM translations processing

In [ ]:
df_gptmini = pd.read_pickle('data/gptmini_translations.pkl')

# keep only rows where en_norm freq > 5
en_norm_freq = df_gptmini['en_norm'].value_counts()
frequent_en_norm = en_norm_freq[en_norm_freq > 5].index
df_gptmini = df_gptmini[df_gptmini['en_norm'].isin(frequent_en_norm)]

de_norm_freq = df_gptmini['de_norm'].value_counts()
frequent_de_norm = de_norm_freq[de_norm_freq > 5].index
df_gptmini = df_gptmini[df_gptmini['de_norm'].isin(frequent_de_norm)]

sr_norm_freq = df_gptmini['sr_norm'].value_counts()
frequent_sr_norm = sr_norm_freq[sr_norm_freq > 5].index
df_gptmini = df_gptmini[df_gptmini['sr_norm'].isin(frequent_sr_norm)]

# remove rows where any of the norms is None or empty
df_gptmini = df_gptmini[
    df_gptmini['en_norm'].notna() & (df_gptmini['en_norm'] != '') & (df_gptmini['en_norm'] != 'None') &
    df_gptmini['de_norm'].notna() & (df_gptmini['de_norm'] != '') & (df_gptmini['de_norm'] != 'None') &
    df_gptmini['sr_norm'].notna() & (df_gptmini['sr_norm'] != '') & (df_gptmini['sr_norm'] != 'None')
]
print(f"Size after filtering: {len(df_gptmini)}")

# merge with aligned_frequent on hash
df_gptmini = pd.merge(aligned_frequent[[ 'hash', 'fr_norm', 'embeddings']], df_gptmini, on='hash', how='inner')


In [ ]:
# predicting similarities for gptmini data
size = len(df_gptmini)
predicted_sim_gptmini = np.zeros((size, size))
for i in range(size):
    for j in range(i+1, size):
        f_i = df_gptmini.iloc[i]['embeddings']
        f_j = df_gptmini.iloc[j]['embeddings']
        pair = np.array([[f_i, f_j]])
        predicted_sim_gptmini[i, j] = final_pipe.predict(pair)[0]
        predicted_sim_gptmini[j, i] = predicted_sim_gptmini[i, j]

predicted_sim_gptmini_normalized = np.zeros_like(predicted_sim_gptmini)
for i in range(size):
    predicted_sim_gptmini_normalized[i] = softmax(predicted_sim_gptmini[i])

# does each row sum to 1? boolean check
row_sums_gptmini = predicted_sim_gptmini_normalized.sum(axis=1)
print("All rows sum to 1 (GPT-Mini):", np.allclose(row_sums_gptmini, 1.0))

In [ ]:
# alignement tables from df_gptmini
pairs = [('fr', 'en'), ('fr', 'de'), ('fr', 'sr')]
gpt_tables = []
gpt_dataframes = []
for source, target in pairs:
    lexicon = df_gptmini[f'{target}_norm'].unique()
    source_terms = df_gptmini[f'{source}_norm']
    ids = df_gptmini.index
    source_hash = [f"{term}_{ID}" for ID,term in zip(ids, source_terms)]
    
    translations = np.zeros((len(df_gptmini), len(lexicon)), dtype=int)
    print(translations.shape)
    for i,row in df_gptmini.iterrows():
        target_term = row[f'{target}_norm']
        if target_term in lexicon:
            j = lexicon.tolist().index(target_term)
            translations[i,j] += 1
        else :
            print(f"Skipping target term: {target_term}")
            pass
    gpt_tables.append(translations)
    gpt_dataframes.append(pd.DataFrame(translations, columns=lexicon))

# assert each row sums to 1
for df in gpt_dataframes:
    assert all(df.sum(axis=1) == 1)

# 4) IB analysis

## locating attested and generated translations in the information plane

In [ ]:
PRECISION = 1e-16

def xlogx(v):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(v > PRECISION, v * np.log2(v), 0)

def H(p, axis=None):
    """ Entropy in bits """
    return -xlogx(p).sum(axis=axis)

def MI_bits(pXY):
    """ mutual information, I(X;Y), in bits """
    return H(pXY.sum(axis=0)) + H(pXY.sum(axis=1)) - H(pXY)

def complexity_bits( pW_M, pM):
    """
    p_y_x : Conditional distribution on Y given X, of shape X x Y.
    :param pW_M: encoder (naming system)
    :return: I(M;W)
    """
    return MI_bits(pW_M * pM)

def accuracy_bits(pW_M, pM, pU_M):
    """
    :param pW_M: encoder (naming system)
    :return: I(W;U)
    """
    pMW = pW_M * pM
    pWU = pMW.T @ pU_M
    return MI_bits(pWU)
    
def marginalize(pY_X, pX):
    """:return  pY """
    return pY_X.T @ pX

def information_plane(p_x, p_y_x, p_z_x):
    """ Given p(x), p(y|x), and p(z|x), calculate I[Y:Z] and I[X:Z]
    given p(m), p(u|m), and p(w|m)
    """
    p_xz = p_x[:, None] * p_z_x # Joint p(x,y), shape X x Y
    p_xyz = p_x[:, None, None] * p_y_x[:, :, None] * p_z_x[:, None, :] # Joint p(x,y,z), shape X x Y x Z
    p_yz = p_xyz.sum(axis=0) # Joint p(y,z), shape Y x Z
    informativity = MI_bits(p_yz)
    complexity = MI_bits(p_xz)
    
    return informativity, complexity

In [ ]:
uniform_prob = 1 / predicted_sim_normalized.shape[0]
p_m = np.asarray([uniform_prob]*predicted_sim_normalized.shape[0], dtype=float)
INFOS = []
COMPS = []
for translations in tables:
    COMPS.append(complexity_bits(translations, p_m[:, None]))
    INFOS.append(accuracy_bits(translations, p_m[:, None], predicted_sim_normalized))

# LLM translations
uniform_prob_gptmini = 1 / predicted_sim_gptmini_normalized.shape[0]
p_m_gptmini = np.asarray([uniform_prob_gptmini]*predicted_sim_gptmini_normalized.shape[0], dtype=float)
INFOS_gptmini = []
COMPS_gptmini = []
for translations in gpt_tables:
    COMPS_gptmini.append(complexity_bits(translations, p_m_gptmini[:, None]))
    INFOS_gptmini.append(accuracy_bits(translations, p_m_gptmini[:, None], predicted_sim_gptmini_normalized))

plt.figure(figsize=(4, 2))
plt.scatter(COMPS, INFOS, color='blue')
plt.scatter(COMPS_gptmini, INFOS_gptmini, color='orange')
for comp, info, pair in zip(COMPS, INFOS, pairs):
    plt.text(comp, info, f'->{pair[1]}', fontsize=6, ha='left')
for comp, info, pair in zip(COMPS_gptmini, INFOS_gptmini, pairs):
    plt.text(comp, info, f'->{pair[1]} (GPT-Mini)', fontsize=6, ha='left')
plt.xlabel('Complexity I(M;W)')
plt.ylabel('Informativity I(W;U)')
plt.title('Information Plane of Language Pairs')
plt.grid(False)
plt.show()

## counter-factual translations

`N_RANDOM` is the number of fully random counter-factual translations (set to 100 for speed of computation)

In [ ]:
N_RANDOM = 100

def random_Translations(translations) -> np.ndarray:
    """
    generate a random matrix of shape (M, W) = `translations.shape`
    by randomly generating M one hot encodings of size W,
    where M is the number of rows and W the number of columns of `translations`
    """ 
    M, W = translations.shape
    rand_matrix = np.zeros((M, W), dtype=int)
    for i in range(M):
        j = np.random.randint(W)
        rand_matrix[i, j] = 1
    return rand_matrix

# Generate random Translations and compute their complexity and informativity
RANDOM_INFOS = []
RANDOM_COMPS = []
for attested in tables:
    p_m = np.asarray([1/attested.shape[0]]*attested.shape[0], dtype=float)
    for _ in range(N_RANDOM):  # Generate N_RANDOM random Translations for each language pair
        random_trans = random_Translations(attested)
        RANDOM_COMPS.append(complexity_bits(random_trans, p_m[:, None]))
        RANDOM_INFOS.append(accuracy_bits(random_trans, p_m[:, None], predicted_sim_normalized))

# mean accuracy and complexity of random Translations
mean_random_info = np.mean(RANDOM_INFOS)
mean_random_comp = np.mean(RANDOM_COMPS)
print(f"Mean Random Informativity: {mean_random_info:.6f}, Mean Random Complexity: {mean_random_comp:.6f}")

# std
std_random_info = np.std(RANDOM_INFOS)
std_random_comp = np.std(RANDOM_COMPS)
print(f"Std Random Informativity: {std_random_info:.6f}, Std Random Complexity: {std_random_comp:.6f}")

plt.figure(figsize=(2, 2))
plt.scatter(RANDOM_COMPS, RANDOM_INFOS, color='lightgray', alpha=0.5, label='Random Translations')
plt.scatter(COMPS, INFOS, color='blue', label='Attested Translations')
plt.xlabel('Complexity I(M;W)')
plt.ylabel('Informativity I(W;U)')
plt.title('Information Plane with Random Translations')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(False)
plt.show()
    

`N_PERMUTATIONS` is the number of random **permutated** counter-factual translations (set to 100 for speed of computation)

In [ ]:
N_PERMUTATIONS = 100

def random_rows_permutations(translations, ratio):
    """
    generate a random matrix of shape (M, W) = `translations.shape`
    by permuting a fraction `ratio` of rows in the original `translation` matrix
    """
    M, W = translations.shape
    n_changes = int(M * ratio)
    row_idxs = np.random.choice(M, n_changes, replace=False)
    permuted_rows = np.random.permutation(row_idxs)
    
    rand_matrix = translations.copy()
    rand_matrix[row_idxs] = translations[permuted_rows]
    
    return rand_matrix

# Generate partially permuted Translations and compute their complexity and informativity
one_p_permutations= []
five_p_permutations = []
ten_p_permutations = []

for table in tables:
    one_p_permutations.append([])
    five_p_permutations.append([])
    ten_p_permutations.append([])
    for _ in range(N_PERMUTATIONS):
        one_p_permutations[-1].append(random_rows_permutations(table, 0.01))
        five_p_permutations[-1].append(random_rows_permutations(table, 0.05))
        ten_p_permutations[-1].append(random_rows_permutations(table, 0.1))

# compute accuracy of each permutation level
one_p_accuracies = []
five_p_accuracies = []
ten_p_accuracies = []

for table, one_p_list, five_p_list, ten_p_list in zip(tables, one_p_permutations, five_p_permutations, ten_p_permutations):
    one_p_accuracies.append([accuracy_bits(p, p_m[:, None], predicted_sim_normalized) for p in one_p_list])
    five_p_accuracies.append([accuracy_bits(p, p_m[:, None], predicted_sim_normalized) for p in five_p_list])
    ten_p_accuracies.append([accuracy_bits(p, p_m[:, None], predicted_sim_normalized) for p in ten_p_list])

## frontier calculation

In [ ]:
DEFAULT_NUM_ITER=10
PRECISION = 1e-16

def _ib(p_x, p_y_x, beta, init, num_iter=DEFAULT_NUM_ITER, temperature = 1):
    """
    Find encoder q(Z|X) to minimize J = I[X:Z] - beta * I[Y:Z].
    Deterministic annealing version of Blahut-Arimoto algorithm for the Information Bottleneck. 

    Input:
    p_x : Distribution on X, of shape X.
    p_y_x : Conditional distribution on Y given X, of shape X x Y.
    beta : A non-negative scalar value.
    Z : Support size of Z.

    Output: 
    Conditional distribution on Z given X, of shape X x Z.
    """

    # Randomly initialize the conditional distribution q(z|x)
    q_z_x = init
    p_y_x = p_y_x[:, None, :] # shape X x 1 x Y
    p_x = p_x[:, None] # shape X x 1

    # Blahut-Arimoto iteration to find the minimizing q(z|x)
    for _ in range(num_iter):
        # Joint distribution q(x,z), shape X x Z
        q_xz = p_x * q_z_x 

        # Marginal distribution q(z), shape 1 x Z
        q_z = q_xz.sum(axis=0, keepdims=True)
        # clip q_z to avoid numerical issues, and renormalize
        q_z = np.clip(q_z, PRECISION, None)
        q_z /= q_z.sum(axis=1, keepdims=True)

        # Conditional decoder distribution q(y|z), shape 1 x Z x Y
        q_y_z = ((q_xz / q_z)[:, :, None] * p_y_x).sum(axis=0, keepdims=True) 
        # negative KL divergence -D[p(y|x) || q(y|z)]
        d = (xlogy(p_y_x, p_y_x) - xlogy(p_y_x, q_y_z)).sum(axis=-1)
        # in bits ;
        d = d / np.log(2)
        
        q_z_x = softmax((np.log2(q_z) - beta*d)/temperature, axis=-1) 

    return q_z_x

`BETAS` log-linearly spaced between 1 and $2^{20}$

In [ ]:
BETAS = np.logspace(20,1,num=100,base=2)

In [ ]:
uniform_prob = 1 / predicted_sim_normalized.shape[0]
p_m = np.asarray([uniform_prob]*predicted_sim_normalized.shape[0], dtype=float)

opti_comp, opti_info = [], []

# Start with a random translator as the initial q(z|x)
q_init = random_Translations(tables[0])  


for beta in tqdm(BETAS, desc="Calculating optimal points", total=len(BETAS)):
    q_beta = _ib(p_m, predicted_sim_normalized, beta, q_init, num_iter = 50)
    temp_info, temp_compl = information_plane(p_m, predicted_sim_normalized, q_beta)
    opti_comp.append(temp_compl)
    opti_info.append(temp_info)
    q_init = q_beta
    print(f"beta: {beta}, informativity: {temp_info}, complexity: {temp_compl}")

curve = pd.DataFrame(
    data = {
        'beta': BETAS,
        'informativity' : opti_info,
        'complexity' : opti_comp,
        'J' : np.array(opti_comp) - np.array(BETAS)*np.array(opti_info)
        }
)

plt.figure(figsize=(5,5))
plt.plot(curve["complexity"], curve["informativity"], color='red', label='Optimal IB Frontier')
plt.scatter(COMPS, INFOS, color='blue', label='Language Pairs')
plt.scatter(COMPS_gptmini, INFOS_gptmini, color='orange', label='GPT-Mini Translations')
plt.xlabel('Complexity')
plt.ylabel('Informativity')
plt.legend()
plt.show()


## $\epsilon$ distance to optimal encoder

In [ ]:
# function to calculate efficiency loss (Zaslavsky et al., 2018)
def find_epsilon(p_m, p_u_m, q_w_m, curve_complexity, curve_informativity, BETAS):
    Fb = complexity_bits(q_w_m, p_m[:, None]) - BETAS * accuracy_bits(q_w_m, p_m[:, None], p_u_m)
    Fb_star= curve_complexity - BETAS * curve_informativity
    diff = Fb - Fb_star
    return(diff.min()/BETAS[diff.argmin()])

In [ ]:
BETAS = np.logspace(20,1,num=100,base=2)

p_u_m = predicted_sim_normalized
attested_epsilon = []
for q_w_m,pair in zip(tables, pairs):
    p_m = np.asarray([1 / q_w_m.shape[0]] * q_w_m.shape[0], dtype=float)
    epsilon = find_epsilon(
        p_m,
        predicted_sim_normalized,
        q_w_m,
        curve["complexity"],
        curve["informativity"],
        BETAS
    )
    print(f"Efficiency loss (epsilon) for pair {pair}: {epsilon:.6f}")
    attested_epsilon.append(epsilon)

In [ ]:
# Compute epsilon for all pair and all perturbation levels
EN_DEVIATIONS_ONE = []
for fake_encoder in one_p_permutations[0] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    EN_DEVIATIONS_ONE.append(epsilon)

EN_DEVIATIONS_FIVE = []
for fake_encoder in five_p_permutations[0] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    EN_DEVIATIONS_FIVE.append(epsilon)

EN_DEVIATIONS_TEN = []
for fake_encoder in ten_p_permutations[0] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    EN_DEVIATIONS_TEN.append(epsilon)

DE_DEVIATIONS_ONE = []
for fake_encoder in one_p_permutations[1] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    DE_DEVIATIONS_ONE.append(epsilon)
DE_DEVIATIONS_FIVE = []
for fake_encoder in five_p_permutations[1] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    DE_DEVIATIONS_FIVE.append(epsilon)
DE_DEVIATIONS_TEN = []
for fake_encoder in ten_p_permutations[1] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    DE_DEVIATIONS_TEN.append(epsilon)

SR_DEVIATIONS_ONE = []
for fake_encoder in one_p_permutations[2] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    SR_DEVIATIONS_ONE.append(epsilon)
SR_DEVIATIONS_FIVE = []
for fake_encoder in five_p_permutations[2] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    SR_DEVIATIONS_FIVE.append(epsilon)
SR_DEVIATIONS_TEN = []
for fake_encoder in ten_p_permutations[2] :
    p_m = np.asarray([1 / fake_encoder.shape[0]] * fake_encoder.shape[0], dtype=float)
    epsilon = np.abs(find_epsilon(p_m, predicted_sim_normalized, fake_encoder, curve["complexity"], curve["informativity"], BETAS))
    SR_DEVIATIONS_TEN.append(epsilon)

In [ ]:
# deviation of random translators
RANDOM_TRANSLATIONS_DEVIATIONS = []
for random_comp, random_info in zip(RANDOM_COMPS, RANDOM_INFOS):
    Fb = random_comp - BETAS * random_info
    Fb_star= curve['complexity'] - BETAS * curve['informativity']
    diff = Fb - Fb_star
    eps = diff.min()/BETAS[diff.argmin()]
    RANDOM_TRANSLATIONS_DEVIATIONS.append(eps)

# split in 3 lists according to the pair
RANDOM_EN_DEVIATIONS = RANDOM_TRANSLATIONS_DEVIATIONS[:len(one_p_permutations[0])]
RANDOM_DE_DEVIATIONS = RANDOM_TRANSLATIONS_DEVIATIONS[len(one_p_permutations[0]):len(one_p_permutations[0])+len(one_p_permutations[1])]
RANDOM_SR_DEVIATIONS = RANDOM_TRANSLATIONS_DEVIATIONS[len(one_p_permutations[0])+len(one_p_permutations[1]):]

In [ ]:
gpt_mini_deviations = []
for q_w_m in gpt_tables:
    p_m = np.full(q_w_m.shape[0], 1 / q_w_m.shape[0], dtype=float)
    epsilon = find_epsilon(p_m, predicted_sim_gptmini_normalized, q_w_m, curve["complexity"], curve["informativity"], BETAS)
    gpt_mini_deviations.append(epsilon)

In [ ]:
# dict for plotting
means = {
    'Attested': attested_epsilon,
    '1%': [np.mean(EN_DEVIATIONS_ONE), np.mean(DE_DEVIATIONS_ONE), np.mean(SR_DEVIATIONS_ONE)],
    '5%': [np.mean(EN_DEVIATIONS_FIVE), np.mean(DE_DEVIATIONS_FIVE), np.mean(SR_DEVIATIONS_FIVE)],
    '10%': [np.mean(EN_DEVIATIONS_TEN), np.mean(DE_DEVIATIONS_TEN), np.mean(SR_DEVIATIONS_TEN)],
}
stds = {
    'Attested': [0.0]*3,
    '1%': [np.std(EN_DEVIATIONS_ONE), np.std(DE_DEVIATIONS_ONE), np.std(SR_DEVIATIONS_ONE)],
    '5%': [np.std(EN_DEVIATIONS_FIVE), np.std(DE_DEVIATIONS_FIVE), np.std(SR_DEVIATIONS_FIVE)],
    '10%': [np.std(EN_DEVIATIONS_TEN), np.std(DE_DEVIATIONS_TEN), np.std(SR_DEVIATIONS_TEN)],
}

x = np.arange(len(pairs))
width = 0.15
original_col = '#009E73'
colors = ['#D55E00', '#0072B2', '#E69F00']


fig, ax = plt.subplots()

# attested translations
ax.bar(x - width, means['Attested'], width, label='Attested translations', color=original_col)

# permutated translations
ax.bar(x, means['1%'], width, yerr=stds['1%'], capsize=2, label='1% Perturbation', color=colors[0])
ax.bar(x + width, means['5%'], width, yerr=stds['5%'], capsize=2, label='5% Perturbation', color=colors[1])
ax.bar(x + 2*width, means['10%'], width, yerr=stds['10%'], capsize=2, label='10% Perturbation', color=colors[2])

# random translations
ax.bar(x + 3*width, 
       [np.mean(RANDOM_EN_DEVIATIONS), np.mean(RANDOM_DE_DEVIATIONS), np.mean(RANDOM_SR_DEVIATIONS)],
       width,
       yerr=[np.std(RANDOM_EN_DEVIATIONS),np.std(RANDOM_DE_DEVIATIONS), np.std(RANDOM_SR_DEVIATIONS)], 
       capsize=2, label='Random translations', color='lightgray'
)


# GPT-Mini translations
ax.bar(x + width*4, gpt_mini_deviations, width,
       label='GPT-Mini translations', color='#CC79A7')

ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(pairs, fontsize=7)
ax.set_ylabel('Deviation from optimality $\epsilon_q$', fontsize=7)
ax.tick_params(axis='y', labelsize=7)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=7)
plt.tight_layout()
plt.show()